In [3]:
# Cell 1 — Environment
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch

# M3 Mac device check
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")  # should print: mps

Using device: mps


In [1]:
# Cell 0 — sanity check before loading
import subprocess
result = subprocess.run(["sysctl", "hw.memsize"], capture_output=True, text=True)
ram_gb = int(result.stdout.split(":")[1].strip()) / 1e9
print(f"Total RAM: {ram_gb:.1f} GB")
# Need at least 16GB — 32GB recommended

Total RAM: 19.3 GB


In [4]:
# Cell 2 — Load model (NO bitsandbytes on Mac)
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path

base_model_id = "Qwen/Qwen2.5-7B-Instruct"  # ✅ HF accessible in Canada
adapter_path  = "../outputs/jinyong-qlora/adapter"  # adjust to your local path

# ✅ Load tokenizer from adapter folder (already saved there)
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ✅ No quantization on Mac — use bfloat16 instead
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,   # bfloat16 also works on M3
    device_map="cpu",            # load to CPU first, then move to MPS
    trust_remote_code=True,
)

# Load LoRA adapter
model = PeftModel.from_pretrained(model, adapter_path)
model = model.to(device)        # move to MPS
model.eval()
print("✅ Model ready on", device)

Loading checkpoint shards: 100%|██████████| 4/4 [00:50<00:00, 12.73s/it]
/Users/william.jiang/my-tests/jinyong-finetune/venv/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'
✅ Model ready on mps


In [ ]:
import os
cache = os.path.expanduser("~/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots")
snapshot = os.listdir(cache)[0]
base_model_local = os.path.join(cache, snapshot)
# Then use base_model_local instead of "Qwen/Qwen2.5-7B-Instruct"

In [5]:
# Cell 3 — Inference
messages = [
    {"role": "system", "content": "你是金庸小说里的郭靖，用郭靖的口吻和性格回答问题。"},
    {"role": "user",   "content": "你如何看待江湖恩怨？"}
]

text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)
print(response)

: 